# ĐỀ TÀI: PHÂN TÍCH BIẾN ĐỘNG VÀ CHÊNH LỆCH GIÁ VÀNG SJC BẰNG PYTHON
* **Môn học:** Phân tích dữ liệu Python
* **Nhóm thực hiện:** Nhóm 12
* **Sổ tay:** 01_data_cleaning_eda.ipynb (Làm sạch dữ liệu & phân tích khám phá ban đầu)
* **Mục tiêu giai đoạn:** Hoàn thiện làm sạch dữ liệu, ghép bảng đạt >= 300 bản ghi và xuất thống kê mô tả ban đầu (Mốc 03 - 10/11/2026).

In [1]:
# 1. Khai báo các thư viện nền tảng
import pandas as pd
import numpy as np
import os

# Cấu hình hiển thị định dạng số thực cho rõ ràng (tránh bị hiển thị dạng 1.4e+07)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Thư viện đã sẵn sàng!")

Thư viện đã sẵn sàng!


In [2]:
def clean_sjc(filepath: str) -> pd.DataFrame:
    """ Quy trình đọc và làm sạch dữ liệu SJC:
    - Loại bỏ ký tự BOM ẩn (\ufeff)
    - Chuẩn hóa tên cột thành 'ngay', 'sjc_mua', 'sjc_ban'
    - Ép kiểu datetime cho cột 'ngay'
    - Ép kiểu số thực (float) cho 2 cột giá
    - Loại bỏ dòng khuyết thiếu (NaN) và dòng trùng lặp
    """
    # Đọc file CSV thô
    df = pd.read_csv(filepath)
    
    # Xóa ký tự BOM và khoảng trắng thừa ở tiêu đề cột
    df.columns = [col.strip().replace('\ufeff', '') for col in df.columns]
    
    # Chuẩn hóa tên cột
    rename_mapping = {}
    for col in df.columns:
        c_low = col.lower()
        if c_low in ['date', 'ngày', 'ngay']:
            rename_mapping[col] = 'ngay'
        elif 'mua' in c_low:
            rename_mapping[col] = 'sjc_mua'
        elif 'ban' in c_low or 'bán' in c_low:
            rename_mapping[col] = 'sjc_ban'
            
    df = df.rename(columns=rename_mapping)
    
    # Chuyển đổi định dạng ngày tháng và sắp xếp theo thời gian
    df['ngay'] = pd.to_datetime(df['ngay'], errors='coerce')
    df = df.sort_values('ngay').reset_index(drop=True)
    
    # Ép kiểu dữ liệu dạng số thực cho giá mua và bán
    df['sjc_mua'] = pd.to_numeric(df['sjc_mua'], errors='coerce')
    df['sjc_ban'] = pd.to_numeric(df['sjc_ban'], errors='coerce')
    
    # Loại bỏ các dòng trống/thiếu dữ liệu (ví dụ ngày 01/01/2020)
    df = df.dropna(subset=['ngay', 'sjc_mua', 'sjc_ban'])
    
    # Loại bỏ dòng trùng lặp theo ngày
    df = df.drop_duplicates(subset=['ngay']).reset_index(drop=True)
    
    return df[['ngay', 'sjc_mua', 'sjc_ban']]

In [3]:
# Gọi hàm làm sạch 
# Lưu ý: nếu notebook chạy từ thư mục 'notebooks/', dùng đường dẫn '../data/raw/sjc_raw.csv' 
path_sjc = '../data/raw/sjc_raw.csv' if os.path.exists('../data/raw/sjc_raw.csv') else 'data/raw/sjc_raw.csv' 
df_sjc = clean_sjc(path_sjc) 

# In báo cáo kiểm tra kỹ thuật 
print("================ KẾT QUẢ LÀM SẠCH DỮ LIỆU SJC ================") 
print(f"1. Tổng số bản ghi hợp lệ: {len(df_sjc):,} dòng") 
print(f"2. Danh sách cột chuẩn: {list(df_sjc.columns)}") 
print(f"3. Kiểu dữ liệu từng cột:\n{df_sjc.dtypes}") 
print(f"4. Số lượng giá trị Null còn sót lại:\n{df_sjc.isnull().sum()}") 
print(f"5. Phạm vi thời gian: từ {df_sjc['ngay'].min().strftime('%Y-%m-%d')} đến {df_sjc['ngay'].max().strftime('%Y-%m-%d')}") 
print("==============================================================")

================ KẾT QUẢ LÀM SẠCH DỮ LIỆU SJC ================
1. Tổng số bản ghi hợp lệ: 1,782 dòng
2. Danh sách cột chuẩn: ['ngay', 'sjc_mua', 'sjc_ban']
3. Kiểu dữ liệu từng cột:
ngay       datetime64[us]
sjc_mua           float64
sjc_ban           float64
dtype: object
4. Số lượng giá trị Null còn sót lại:
ngay       0
sjc_mua    0
sjc_ban    0
dtype: int64
5. Phạm vi thời gian: từ 2020-01-02 đến 2026-10-30


In [4]:
# Hiển thị 5 dòng đầu tiên (đảm bảo dòng lỗi NaN ngày 2020-01-01 đã được xóa bỏ) 
print("5 dòng đầu tiên:") 
df_sjc.head()

5 dòng đầu tiên:


,ngay,sjc_mua,sjc_ban
0,2020-01-02,40690000.00,42710000.00
1,2020-01-03,41570000.00,43070000.00
2,2020-01-06,41950000.00,43460000.00
3,2020-01-07,41340000.00,43060000.00
4,2020-01-08,41650000.00,43190000.00


In [5]:
# Hiển thị 5 dòng gần nhất 
print("5 dòng gần nhất:") 
df_sjc.tail()

5 dòng gần nhất:


,ngay,sjc_mua,sjc_ban
1777,2026-10-26,138410000.00,142560000.00
1778,2026-10-27,138820000.00,142230000.00
1779,2026-10-28,137760000.00,141770000.00
1780,2026-10-29,138300000.00,141920000.00
1781,2026-10-30,137390000.00,141250000.00
